# 0. Data preparation

In [ ]:
import os
import re
import sys
import argparse
import subprocess
import matplotlib
import time
import random
import string
import shlex
import shutil
import glob
import pickle
import csv
import operator
import joblib
import pandas as pd
import numpy as np
from itertools import groupby
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import precision_recall_curve
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from collections import defaultdict
from sklearn.ensemble import BaggingClassifier
from sklearn import svm
from sklearn.model_selection import StratifiedKFold
from Bio import SeqIO
from sklearn.feature_selection import RFE
from sklearn import preprocessing
import pybedtools as pbt
import pyBigWig as pbw
from datetime import date
from xgboost import XGBClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from itertools import chain
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from sklearn.ensemble import AdaBoostClassifier
from adjustText import adjust_text
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from pylab import *
%matplotlib inline

In [ ]:
#os.chdir("/home/pavelkrav/")

root = os.getcwd()
basicdir = os.path.abspath('Release/TF-ML/')
outputdir = os.path.abspath('Release/TF-ML/outputdir')
train_dir = os.path.abspath('Release/TF-ML/train/') 
test_dir = os.path.abspath('Release/TF-ML/test/') 

if not os.path.exists(basicdir):
    os.makedirs(basicdir)
if not os.path.exists(outputdir):
    os.makedirs(outputdir)
if not os.path.exists(train_dir):
    os.makedirs(train_dir)
if not os.path.exists(test_dir):
    os.makedirs(test_dir)

## Experiment and TF selection

In [ ]:
metadata_experiments = pd.read_csv("./hocomoco11/metadata_23_02_2017.txt", sep="\t")
metadata_experiments
metadata_experiments_sel = metadata_experiments[["exp","specie","TFclass","control_id","Uniprot_AC","gem_peaks","peak_id"]]
metadata_experiments_sel = metadata_experiments_sel.fillna("abrakadabra")
metadata_experiments_sel_control = metadata_experiments_sel[metadata_experiments_sel["control_id"] != "abrakadabra"]
metadata_experiments_sel_control

In [ ]:
bench_mono_df = pd.read_csv("./hocomoco11/curation/slices4bench_mono/out_sure.csv", header=None)
bench_mono = [x[:-1] for x in bench_mono_df[0].str.split(".")]

cond = []

for i in range(metadata_experiments_sel_control.shape[0]):
    df = metadata_experiments_sel_control.iloc[i,]
    if [df["Uniprot_AC"], df["peak_id"], "macs"] in bench_mono:
        cond.append(True)
    else:
        cond.append(False)
 
metadata_TF_filter_1 = metadata_experiments_sel_control[cond]
metadata_TF_filter_1.shape[0]

print("Good datasets with the controls:", metadata_TF_filter_1.drop_duplicates(subset=["Uniprot_AC"]).shape)

In [ ]:
metadata_experiments_sel_control_M = metadata_TF_filter_1[metadata_TF_filter_1["specie"] == "Mus musculus"]
metadata_experiments_sel_control_M = metadata_experiments_sel_control_M.dropna()
metadata_experiments_sel_control_M = metadata_experiments_sel_control_M.groupby(['Uniprot_AC']).count()
metadata_experiments_sel_control_M = metadata_experiments_sel_control_M[metadata_experiments_sel_control_M["exp"] > 2]
slected_TF_names_M = metadata_experiments_sel_control_M.index.values
print(len(slected_TF_names_M))

slected_TF_names_M_s = [x.split("_")[0] for x in slected_TF_names_M]

print(slected_TF_names_M_s, len(slected_TF_names_M_s))

In [ ]:
metadata_experiments_sel_control_H = metadata_experiments_sel_control[metadata_experiments_sel_control["specie"] == "Homo sapiens"]
metadata_experiments_sel_control_H = metadata_experiments_sel_control_H.dropna()
metadata_experiments_sel_control_H = metadata_experiments_sel_control_H.groupby(['Uniprot_AC']).count()
metadata_experiments_sel_control_H = metadata_experiments_sel_control_H[metadata_experiments_sel_control_H["exp"] > 2]
slected_TF_names_H = metadata_experiments_sel_control_H.index.values
print(len(slected_TF_names_H))

slected_TF_names_H_s = [x.split("_")[0] for x in slected_TF_names_H]
print(slected_TF_names_H_s, len(slected_TF_names_H_s))

In [ ]:
intersection_set = set.intersection(set(slected_TF_names_H_s), set(slected_TF_names_M_s))
intersection_list = list(intersection_set)
len(intersection_list)
#intersection_list
# 42 TFs in total intersection

In [ ]:
metadata_TF_filter_1["Uniprot_name"] = [x[0] for x in metadata_TF_filter_1["Uniprot_AC"].str.split("_")]

In [ ]:
metadata_TF_filter_2 = metadata_TF_filter_1[[x in intersection_list for x in metadata_TF_filter_1["Uniprot_name"]]]

In [ ]:
intersection_list = sorted(intersection_list)
intersection_list

## Preparing HUMAN data

In [ ]:
organism = "HUMAN"
caller = "macs"

slices4bench_mono = [i for i in pd.read_csv("./hocomoco11/curation/slices4bench_mono/out_sure.csv", header=None)[0] if caller in i]
files_wlogauc_datasets = [f for f in os.listdir("./hocomoco11/auc/mono/HUMAN_datasets/")]
print(len(files_wlogauc_datasets))
files_wlogauc_datasets_sel = sorted([x for x in files_wlogauc_datasets if ((x.split("_")[0] in intersection_list) & (organism in x) & (x.split("~")[-1].split(".txt")[0] in slices4bench_mono))])
print(len(files_wlogauc_datasets_sel))
files_wlogauc_datasets_sel = [x for x in files_wlogauc_datasets_sel if x.split(".")[1] in list(metadata_TF_filter_2["peak_id"])]
print(len(files_wlogauc_datasets_sel))

d = {}
for i in files_wlogauc_datasets_sel:
    if i.split("_")[0] in d.keys():
        d[i.split("_")[0]] = d[i.split("_")[0]] + [i]
    else:
        d[i.split("_")[0]] = [i]

d_datasets = {}
for i in d.keys():
    print(i)
    tmp = []
    print("datasets:", len(d[i]))
    for j in d[i]:
        if caller in j:
            f = pd.read_csv("./hocomoco11/auc/mono/"+organism+"_datasets/"+j, sep="\t", header=None)
            series = f[0]
            f = f.sort_values(by=[1], ascending=False)
            p10 = np.percentile(f[1], 10)
            if p10> 0.5:
                tmp.append(j)

    print("peak files:", len(tmp))
    tmp = list(set(tmp))
    print("peak files:", len(tmp))
    if len(tmp)>0:
        d_datasets[i] = tmp

In [ ]:
d_datasets_H = d_datasets

In [ ]:
d_datasets.keys(), len(d_datasets.keys())
# 39 TFs survived

In [ ]:
def find_peaks(df_in):
    df_tmp = pd.DataFrame(columns=list(pd.read_csv("/home/pavelkrav/macs/PEAKS041820.interval", index_col=None, sep="\t").columns)+["ind"])
    window = 1000
    df_tmp["abs_summit"] = df_in["START"] + df_in["summit"]
    df_tmp["START"] = df_in["START"] + df_in["summit"] - window//2
    df_tmp[df_tmp["START"] < 0] = 0
    df_tmp["END"] = df_in["START"] + df_in["summit"] + window//2 
    df_tmp = df_tmp.drop(columns=["summit","tags","FDR(%)","fold_enrichment"])
    df_tmp['#CHROM'] = df_in['#CHROM'].apply(lambda t : f"chr{t}")
    df_tmp = df_tmp[(df_tmp["#CHROM"] != 'chrnan') & (df_tmp["#CHROM"] != 'chrM') & (df_tmp["#CHROM"].str.len() < 6)]
    df_tmp["-10*log10(pvalue)"] = df_in["-10*log10(pvalue)"] 
    df_tmp = df_tmp[["#CHROM", "START", "END", "-10*log10(pvalue)", "abs_summit"]]
    return df_tmp
                        
def repeats_cleaner(df_in):
    replic = pbt.BedTool.from_dataframe(df_in)
    replic.count()

    repeats = pbt.BedTool("/home/pavelkrav/Cleaning_data/track_out.bed")

    without_reps = replic.subtract(repeats, f=0.7, N=True)
    without_reps.count()
    
    return without_reps.to_dataframe()

# Human data prepararion

In [ ]:
d_peaks = {}
d_peaks_all = {}

for TF in d_datasets.keys():
    print(TF, len(d_datasets[TF]))
    tmp = pd.DataFrame()
    for i in d_datasets[TF]:
        peak_n = i.split(".")[1]
        peak_n = "/home/pavelkrav/macs/" + peak_n + ".interval"
        try:
            df = pd.read_csv(peak_n, sep="\t")
            df = df[df["tags"] >= 10]
            df = df.sort_values(by=["-10*log10(pvalue)"], ascending=False)
            df = df.iloc[:1000]
            tmp = pd.concat([tmp, df], ignore_index = True)
        except FileNotFoundError:
            continue
    if tmp.shape[0] > 0:
        TF_peaks_bed = find_peaks(tmp)
        TF_peaks_bed_cleaned = repeats_cleaner(TF_peaks_bed)
        d_peaks[TF] = TF_peaks_bed_cleaned
        d_peaks_all[TF] = TF_peaks_bed

In [ ]:
for i in d_peaks.keys():
    print(i, d_peaks[i].shape[0])

In [ ]:
train_chr = ["chr"+x for x in ["2", "3", "4", "5", "6", "7", "9", "10", "11" "12", "13", "14", "15", "16", "17", "18", "19", "20"]]
test_chr = ["chr"+x for x in ["1", "8", "21"]]
print(test_chr)
train_median = []
test_median = []

d_peks_train_test = {}
for TF in d_peaks.keys():
    train_set = d_peaks[TF][[True if x in train_chr else False for x in list(d_peaks[TF]["chrom"])]]
    test_set = d_peaks[TF][[True if x in test_chr else False for x in list(d_peaks[TF]["chrom"])]]
    
    if train_set.shape[0] > 10000:
        train_set = train_set.sample(n=10000, random_state=1)
    if test_set.shape[0] > 10000:
        test_set = test_set.sample(n=10000, random_state=1)

    print(TF, train_set.shape[0], test_set.shape[0])
    train_median.append(np.max(train_set["name"]))
    train_set["name"] = train_set[["name", "score"]].apply(lambda row: ':'.join(row.values.astype(str)), axis=1)
    test_median.append(np.max(test_set["name"]))
    test_set["name"] = test_set[["name", "score"]].apply(lambda row: ':'.join(row.values.astype(str)), axis=1)
    train_set["factor"] = TF
    test_set["factor"] = TF
    d_peks_train_test[TF] = [train_set, test_set]

In [ ]:
sns.distplot(np.array(train_median)/np.array(test_median))

In [ ]:
fasta = pbt.example_filename('/home/pavelkrav/Genomic_data/hg38_genome.fasta')

all_bed_df_train = pd.DataFrame()
all_bed_df_test = pd.DataFrame()

d_fasta = {}
for TF in d_peks_train_test.keys():
    print(TF)
    tmp = d_peks_train_test[TF][0]
    all_bed_df_train = pd.concat([all_bed_df_train, tmp], ignore_index=True)
    d_peks_train_test[TF][0].to_csv(f"{basicdir}/train/{TF}_train_set_HUMAN.bed",  sep='\t')
    train_peaks = pbt.BedTool.from_dataframe(d_peks_train_test[TF][0])
    train_fasta = train_peaks.sequence(fi=fasta, name=True)
    tmp = d_peks_train_test[TF][1]
    all_bed_df_test = pd.concat([all_bed_df_test, tmp], ignore_index=True)
    d_peks_train_test[TF][1].to_csv(f"{basicdir}/test/{TF}_test_set_HUMAN.bed",  sep='\t')
    test_peaks = pbt.BedTool.from_dataframe(d_peks_train_test[TF][1])
    test_fasta = test_peaks.sequence(fi=fasta, name=True)
    d_fasta[TF] = [train_fasta, test_fasta]    

In [ ]:
all_bed_df_train.to_csv(f"{basicdir}/all_bed_df_train_set_HUMAN.bed",  sep='\t')
all_bed_df_test.to_csv(f"{basicdir}/all_bed_df_test_set_HUMAN.bed",  sep='\t')

In [ ]:
selected_HUMAN = [x + "_HUMAN" for x in d_fasta.keys()]

In [ ]:
f = open(d_fasta[TF][0].seqfn).read()

files_mfa_HUMAN_train = []
files_mfa_HUMAN_test = []

for TF in d_fasta.keys():
    # train
    f_name = f"{basicdir}/train/" + TF + "_HUMAN.PEAKS000001.macs.train.mfa"
    print(f_name)
    a = d_fasta[TF][0]  
    a.save_seqs(f_name)
    files_mfa_HUMAN_train.append(f_name)

    # test
    f_name = f"{basicdir}/test/" + TF + "_HUMAN.PEAKS000001.macs.test.mfa"
    print(f_name)
    a = d_fasta[TF][1] 
    a.save_seqs(f_name)
    files_mfa_HUMAN_test.append(f_name)

In [ ]:
size_of_set = 10000

def fasta_iter(f):
    faiter = (x[1] for x in groupby(f, lambda line: line[0] == ">"))

    for header in faiter:
        headerStr = header.__next__()[1:].strip()
        seq = "".join(s.strip() for s in faiter.__next__())
        yield (headerStr, seq)


outfile_name_HUMAN_train = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_train.fasta"
outfile_name_HUMAN_control = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_control.fasta"

outfile_HUMAN_train = open(outfile_name_HUMAN_train, "w")  
outfile_HUMAN_control = open(outfile_name_HUMAN_control, "w")  
log_files_to_mfa = open("log_files_to_mfa.txt", "w") 

headerStr_names_outfile_HUMAN_train = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_HUMAN_control = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_control.csv"

def my_function(x):
    return list(dict.fromkeys(x))

headerStr_names_HUMAN_train = []
index_global = 0

for i in sorted(selected_HUMAN):
    print(i)
    tmp_file_list = []
    for j in files_mfa_HUMAN_train:
        if i in j:
            headerStr_base = "_".join(j.split(".")[:-1]).split("/")[-1]
            print(headerStr_base)
            with open(j, "r") as f:
                fiter = fasta_iter(f)
                index_local = 0
                for seq_rec in fiter:
                    headerStr_new = headerStr_base + "_globalInd_" + str(index_global) + "_localInd_" + str(index_local) + "_train"
                    headerStr, seq = seq_rec
                    print(len(seq))
                    seq = seq[350:650]
                    print(len(seq))
                    if len(seq) < 320 and len(seq) > 280:
                        seq = seq[(len(seq)//2)-150:(len(seq)//2)+150]
                        print(len(seq))
                    break
    break

In [ ]:
size_of_set = 10000

def fasta_iter(f):
    faiter = (x[1] for x in groupby(f, lambda line: line[0] == ">"))

    for header in faiter:
        headerStr = header.__next__()[1:].strip()
        seq = "".join(s.strip() for s in faiter.__next__())
        yield (headerStr, seq)


outfile_name_HUMAN_train = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_train.fasta"
outfile_name_HUMAN_control = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_control.fasta"


outfile_HUMAN_train = open(outfile_name_HUMAN_train, "w")  
outfile_HUMAN_control = open(outfile_name_HUMAN_control, "w")  
log_files_to_mfa = open("log_files_to_mfa.txt", "w") 

headerStr_names_outfile_HUMAN_train = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_HUMAN_control = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_control.csv"

def my_function(x):
    return list(dict.fromkeys(x))

headerStr_names_HUMAN_train = []
index_global = 0

for i in sorted(selected_HUMAN):
    print(i)
    tmp_file_list = []
    for j in files_mfa_HUMAN_train:
        if i in j:
            headerStr_base = "_".join(j.split(".")[:-1]).split("/")[-1]
            print(headerStr_base)
            with open(j, "r") as f:
                fiter = fasta_iter(f)
                index_local = 0
                for seq_rec in fiter:
                    headerStr_new = headerStr_base + "_globalInd_" + str(index_global) + "_localInd_" + str(index_local) + "_train"
                    headerStr, seq = seq_rec
                    seq = seq[350:650]
                    if len(seq) < 320 and len(seq) > 280:
                        seq = seq[(len(seq)//2)-150:(len(seq)//2)+150]
                        if headerStr_base.split("_")[1] == organism:
                            tmp_file_list.append(">" + headerStr_new + "\n" + seq + "\n")
                        index_local += 1
            index_global += 1
    print("tmp_file_list", len(tmp_file_list))

    if len(tmp_file_list) == 0:
        continue

    print("tmp_file_list bewfore ", len(tmp_file_list))
    
    if len(tmp_file_list) > size_of_set:
        tmp_file_list = np.random.choice(tmp_file_list, size_of_set, replace=False, random_state=1)
        
    print("tmp_file_list after", len(tmp_file_list))
    log_files_to_mfa.write(i + " " + str(len(tmp_file_list)) + " " + organism +"_train")
    
    print(len(tmp_file_list))
    tmp_file_list = my_function(tmp_file_list)
    print(len(tmp_file_list))

    for k in tmp_file_list:
        if len(k) != 0:
            headerStr_names_HUMAN_train.append(k[1:].split("\n")[0])
            outfile_HUMAN_train.write(k)

    with open(headerStr_names_outfile_HUMAN_train, "w") as output:
        writer = csv.writer(output, lineterminator='\n')
        for val in headerStr_names_HUMAN_train:
            writer.writerow([val])  

###
print("done!")
headerStr_names_HUMAN_control = []
index_global = 0

for i in sorted(selected_HUMAN):
    print(i)
    tmp_file_list = []
    for j in files_mfa_HUMAN_test:
        if i in j:
            headerStr_base = "_".join(j.split(".")[:-1]).split("/")[-1]
            print(headerStr_base)
            with open(j, "r") as f:
                fiter = fasta_iter(f)
                index_local = 0
                for seq_rec in fiter:
                    headerStr_new = headerStr_base + "_globalInd_" + str(index_global) + "_localInd_" + str(index_local) + "_train"
                    headerStr, seq = seq_rec
                    seq = seq[350:650]
                    if len(seq) < 320 and len(seq) > 280:
                        seq = seq[(len(seq)//2)-150:(len(seq)//2)+150]
                        if headerStr_base.split("_")[1] == organism:
                            tmp_file_list.append(">" + headerStr_new + "\n" + seq + "\n")
                        index_local += 1
            index_global += 1
    print("tmp_file_list", len(tmp_file_list))
    
    if len(tmp_file_list) == 0:
        continue

    print("tmp_file_list bewfore ", len(tmp_file_list))
    if len(tmp_file_list) > size_of_set:
        tmp_file_list = np.random.choice(tmp_file_list, size_of_set, replace=False, random_state=1)
        
    print("tmp_file_list after", len(tmp_file_list))
    log_files_to_mfa.write(i + " " + str(len(tmp_file_list)) + " " + organism +"_control")
    
    print(len(tmp_file_list))
    tmp_file_list = my_function(tmp_file_list)
    print(len(tmp_file_list))
    
    for k in tmp_file_list:
        if len(k) != 0:
            headerStr_names_HUMAN_control.append(k[1:].split("\n")[0])
            outfile_HUMAN_control.write(k)
            
    with open(headerStr_names_outfile_HUMAN_control, "w") as output:
        writer = csv.writer(output, lineterminator='\n')
        for val in headerStr_names_HUMAN_control:
            writer.writerow([val])  

outfile_HUMAN_control.close()
outfile_HUMAN_train.close()
print("done!")

log_files_to_mfa.close()

In [ ]:
size_of_set = 10000


def fasta_iter(f):
    faiter = (x[1] for x in groupby(f, lambda line: line[0] == ">"))

    for header in faiter:
        headerStr = header.__next__()[1:].strip()
        seq = "".join(s.strip() for s in faiter.__next__())
        yield (headerStr, seq)

outfile_name_HUMAN_train = basicdir + "/all_mfa_file_HUMAN_SLIM_" + str(size_of_set) + "_train.fasta"
outfile_name_HUMAN_control = basicdir + "/all_mfa_file_HUMAN_SLIM_" + str(size_of_set) + "_control.fasta"

outfile_HUMAN_train = open(outfile_name_HUMAN_train, "w")  
outfile_HUMAN_control = open(outfile_name_HUMAN_control, "w")  
log_files_to_mfa = open("log_files_to_mfa.txt", "w") 

headerStr_names_outfile_HUMAN_train = basicdir + "/headerStr_names_HUMAN_SLIM_" + str(size_of_set) + "_train.csv" #>chrom: chr1; center: 39934444; peak: 501; signal: 368.13
headerStr_names_outfile_HUMAN_control = basicdir + "/headerStr_names_HUMAN_SLIM_" + str(size_of_set) + "_control.csv" #>chrom: chr1; center: 40382456; peak: 101; signal: 190.21

def my_function(x):
    return list(dict.fromkeys(x))

headerStr_names_HUMAN_train = []
index_global = 0

for i in sorted(selected_HUMAN):
    print(i)
    tmp_file_list = []
    for j in files_mfa_HUMAN_train:
        if i in j:
            headerStr_base = "_".join(j.split(".")[:-1]).split("/")[-1]
            print(headerStr_base)
            with open(j, "r") as f:
                fiter = fasta_iter(f)
                index_local = 0
                for seq_rec in fiter:
                    headerStr, seq = seq_rec
                    headerStr_new = headerStr_base + "_globalInd_" + str(index_global) + "_localInd_" + str(index_local) + "_train"
                    headerStr_new = headerStr_new + "$" + f'>chrom: {headerStr.split(":")[3]}; center: {headerStr.split(":")[1]}; peak: 501; signal: {headerStr.split(":")[0]}'
                    if True:
                        if headerStr_base.split("_")[1] == organism:
                            tmp_file_list.append(">" + headerStr_new + "\n" + seq + "\n")
                        index_local += 1
            index_global += 1
    print("tmp_file_list", len(tmp_file_list))

    if len(tmp_file_list) == 0:
        continue

    print("tmp_file_list bewfore ", len(tmp_file_list))
    
    if len(tmp_file_list) > size_of_set:
        tmp_file_list = np.random.choice(tmp_file_list, size_of_set, replace=False, random_state=1)
        
    print("tmp_file_list after", len(tmp_file_list))
    log_files_to_mfa.write(i + " " + str(len(tmp_file_list)) + " " + organism +"_train")
    
    print(len(tmp_file_list))
    tmp_file_list = my_function(tmp_file_list)
    print(len(tmp_file_list))

    for k in tmp_file_list:
        if len(k) != 0:
            headerStr_names_HUMAN_train.append(k[1:].split("\n")[0])
            outfile_HUMAN_train.write(k)

    with open(headerStr_names_outfile_HUMAN_train, "w") as output:
        writer = csv.writer(output, lineterminator='\n')
        for val in headerStr_names_HUMAN_train:
            writer.writerow([val])  


###
print("done!")
headerStr_names_HUMAN_control = []
index_global = 0

for i in sorted(selected_HUMAN):
    print(i)
    tmp_file_list = []
    for j in files_mfa_HUMAN_test:
        if i in j:
            headerStr_base = "_".join(j.split(".")[:-1]).split("/")[-1]
            print(headerStr_base)
            with open(j, "r") as f:
                fiter = fasta_iter(f)
                index_local = 0
                for seq_rec in fiter:
                    headerStr, seq = seq_rec
                    headerStr_new = headerStr_base + "_globalInd_" + str(index_global) + "_localInd_" + str(index_local) + "_train"
                    headerStr_new = headerStr_new + "$" + f'>chrom: {headerStr.split(":")[3]}; center: {headerStr.split(":")[1]}; peak: 501; signal: {headerStr.split(":")[0]}'
                    if True:
                        if headerStr_base.split("_")[1] == organism:
                            tmp_file_list.append(">" + headerStr_new + "\n" + seq + "\n")
                        index_local += 1
            index_global += 1
    print("tmp_file_list", len(tmp_file_list))
    
    if len(tmp_file_list) == 0:
        continue

    print("tmp_file_list bewfore ", len(tmp_file_list))
    if len(tmp_file_list) > size_of_set:
        tmp_file_list = np.random.choice(tmp_file_list, size_of_set, replace=False, random_state=1)
        
    print("tmp_file_list after", len(tmp_file_list))
    log_files_to_mfa.write(i + " " + str(len(tmp_file_list)) + " " + organism +"_control")
    
    print(len(tmp_file_list))
    tmp_file_list = my_function(tmp_file_list)
    print(len(tmp_file_list))
    
    for k in tmp_file_list:
        if len(k) != 0:
            headerStr_names_HUMAN_control.append(k[1:].split("\n")[0])
            outfile_HUMAN_control.write(k)
            
    with open(headerStr_names_outfile_HUMAN_control, "w") as output:
        writer = csv.writer(output, lineterminator='\n')
        for val in headerStr_names_HUMAN_control:
            writer.writerow([val])  

outfile_HUMAN_control.close()
outfile_HUMAN_train.close()
print("done!")

log_files_to_mfa.close()

In [ ]:
Names = []
Train_list = []

size_of_set = 10000
organism = "HUMAN"

def fasta_iter(f):
    faiter = (x[1] for x in groupby(f, lambda line: line[0] == ">"))

    for header in faiter:
        headerStr = header.__next__()[1:].strip()
        seq = "".join(s.strip() for s in faiter.__next__())
        yield (headerStr, seq)


outfile_name_HUMAN_train = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_train.fasta"
outfile_name_HUMAN_control = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_control.fasta"

headerStr_names_outfile_HUMAN_train = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_HUMAN_control = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_control.csv"

def my_function(x):
    return list(dict.fromkeys(x))

index_global = 0

for i in sorted(selected_HUMAN):
    print(i)
    tmp_file_list = []
    for j in files_mfa_HUMAN_train:
        if i in j:
            headerStr_base = "_".join(j.split(".")[:-1]).split("/")[-1]
            print(headerStr_base)
            with open(j, "r") as f:
                fiter = fasta_iter(f)
                index_local = 0
                for seq_rec in fiter:
                    headerStr_new = headerStr_base + "_globalInd_" + str(index_global) + "_localInd_" + str(index_local) + "_train"
                    headerStr, seq = seq_rec
                    seq = seq[350:650]
                    if len(seq) < 320 and len(seq) > 280:
                        seq = seq[(len(seq)//2)-100:(len(seq)//2)+100]
                        if headerStr_base.split("_")[1] == organism:
                            tmp_file_list.append(">" + headerStr_new + "\n" + seq + "\n")
                        index_local += 1
            index_global += 1
    print("tmp_file_list", len(tmp_file_list))

    if len(tmp_file_list) == 0:
        continue

    print("tmp_file_list bewfore ", len(tmp_file_list))
    
    if len(tmp_file_list) > size_of_set:
        tmp_file_list = np.random.choice(tmp_file_list, size_of_set, replace=False, random_state=1)
        
    print(len(tmp_file_list))
    tmp_file_list = my_function(tmp_file_list)
    print(len(tmp_file_list))

    print("tmp_file_list after", len(tmp_file_list))
    Names.append(i.split("_")[0])
    Train_list.append(len(tmp_file_list))

In [ ]:
print(Names[:10])
print(Train_list[:10])
ddf = pd.DataFrame({"Names":Names,"Seq_count":Train_list})
ddf.to_csv("df_Names_seq_list.csv",  sep='\t')

## Preparing MOUSE data

In [ ]:
organism = "MOUSE"
caller = "macs"

slices4bench_mono = [i for i in pd.read_csv("/home/pavelkrav/hocomoco11/curation/slices4bench_mono/out_sure.csv", header=None)[0] if caller in i]

files_wlogauc_datasets = [f for f in os.listdir("/home/pavelkrav/hocomoco11/auc/mono/"+organism+"_datasets/")]
print(len(files_wlogauc_datasets))
files_wlogauc_datasets_sel = sorted([x for x in files_wlogauc_datasets if ((x.split("_")[0] in intersection_list) & (organism in x) & (x.split("~")[-1].split(".txt")[0] in slices4bench_mono))])
print(len(files_wlogauc_datasets_sel))

d = {}
for i in files_wlogauc_datasets_sel:
    if i.split("_")[0] in d.keys():
        d[i.split("_")[0]] = d[i.split("_")[0]] + [i]
    else:
        d[i.split("_")[0]] = [i]


d_datasets = {}
for i in d.keys():
    print(i)
    tmp = []
    print("datasets:", len(d[i]))
    for j in d[i]:
        if caller in j:
            f = pd.read_csv("/home/pavelkrav/hocomoco11/auc/mono/"+organism+"_datasets/"+j, sep="\t", header=None)
            series = f[0]
            f = f.sort_values(by=[1], ascending=False)
            p10 = np.percentile(f[1], 10)
            if p10> 0.5:
                tmp.append(j)

    tmp = list(set(tmp))
    if len(tmp)>0:
        d_datasets[i] = tmp

In [ ]:
d_datasets_M = d_datasets

In [ ]:
def repeats_cleaner_M(df_in):
    replic = pbt.BedTool.from_dataframe(df_in)
    replic.count()

    repeats = pbt.BedTool("/home/pavelkrav/Cleaning_data/track_out_MOUSE.bed")

    without_reps = replic.subtract(repeats, f=0.7, N=True)
    without_reps.count()
    
    return without_reps.to_dataframe()

In [ ]:
d_peaks = {}
for TF in d_datasets.keys():
    print(TF, len(d_datasets[TF]))
    tmp = pd.DataFrame()
    for i in d_datasets[TF]:
        peak_n = i.split(".")[1]
        peak_n = "/home/pavelkrav/macs/" + peak_n + ".interval"
        try:
            df = pd.read_csv(peak_n, sep="\t")
            df = df[df["tags"] >= 10]
            df = df.sort_values(by=["-10*log10(pvalue)"], ascending=False)
            df = df.iloc[:1000]
            tmp = pd.concat([tmp, df], ignore_index = True)
        except FileNotFoundError:
            continue
    if tmp.shape[0] > 0:
        TF_peaks_bed = find_peaks(tmp)
        TF_peaks_bed_cleaned = repeats_cleaner_M(TF_peaks_bed)
        d_peaks[TF] = TF_peaks_bed_cleaned

In [ ]:
for i in d_peaks.keys():
    print(i, d_peaks[i].shape[0])

In [ ]:
print(d_peaks["ANDR"].head())

train_chr = ["chr"+x for x in ["1", "8", "19"]]
test_chr = ["chr"+x for x in ["2", "3", "4", "5", "6", "7", "9", "10", "11" "12", "13", "14", "15", "16", "17", "18"]]
print(test_chr)

d_peks_train_test = {}
for TF in d_peaks.keys():
    train_set = d_peaks[TF][[True if x in train_chr else False for x in list(d_peaks[TF]["chrom"])]]
    test_set = d_peaks[TF][[True if x in test_chr else False for x in list(d_peaks[TF]["chrom"])]]
    
    if train_set.shape[0] > 10000:
        train_set = train_set.sample(n=10000, random_state=1)
    if test_set.shape[0] > 10000:
        test_set = test_set.sample(n=10000, random_state=1)

    print(TF, train_set.shape[0], test_set.shape[0])
    train_set["name"] = train_set[["name", "score"]].apply(lambda row: ':'.join(row.values.astype(str)), axis=1)
    test_set["name"] = test_set[["name", "score"]].apply(lambda row: ':'.join(row.values.astype(str)), axis=1)
    
    d_peks_train_test[TF] = [train_set, test_set]

In [ ]:
all_bed_df_train = pd.DataFrame()
all_bed_df_test = pd.DataFrame()

#fasta = pbt.example_filename('/home/pavelkrav/Slim/hg38_genome.fasta')
fasta = pbt.example_filename('/home/pavelkrav/Genomic_data/GRCm38.p6.genome.fa')
d_fasta = {}
for TF in d_peks_train_test.keys():
    print(TF)
    tmp = d_peks_train_test[TF][0]
    all_bed_df_train = pd.concat([all_bed_df_train, tmp], ignore_index = True)
    d_peks_train_test[TF][0].to_csv(f"{basicdir}/train/{TF}_train_set_MOUSE.bed",  sep='\t')
    train_peaks = pbt.BedTool.from_dataframe(tmp)
    train_fasta = train_peaks.sequence(fi=fasta, name=True)
    tmp = d_peks_train_test[TF][1]
    all_bed_df_test = pd.concat([all_bed_df_test, tmp], ignore_index = True)
    d_peks_train_test[TF][1].to_csv(f"{basicdir}/test/{TF}_test_set_MOUSE.bed",  sep='\t')
    test_peaks = pbt.BedTool.from_dataframe(d_peks_train_test[TF][1])
    test_fasta = test_peaks.sequence(fi=fasta, name=True)
    d_fasta[TF] = [train_fasta, test_fasta]
                    
selected_MOUSE = [x + "_MOUSE" for x in d_fasta.keys()]

In [ ]:
all_bed_df_train.to_csv(f"{basicdir}/all_bed_df_train_set_MOUSE.bed",  sep='\t')
all_bed_df_test.to_csv(f"{basicdir}/all_bed_df_test_set_MOUSE.bed",  sep='\t')

In [ ]:
f = open(d_fasta[TF][0].seqfn).read()

files_mfa_MOUSE_train = []
files_mfa_MOUSE_test = []

for TF in d_fasta.keys():
    # train
    f_name = f"{basicdir}/train/" + TF + "_"+organism+".PEAKS000001.macs.train.mfa"
    print(f_name)
    a = d_fasta[TF][0]  
    a.save_seqs(f_name)
    files_mfa_MOUSE_train.append(f_name)

    # test
    f_name = f"{basicdir}/test/" + TF + "_"+organism+".PEAKS000001.macs.test.mfa"
    print(f_name)
    a = d_fasta[TF][1]  
    a.save_seqs(f_name)
    files_mfa_MOUSE_test.append(f_name)

In [ ]:
size_of_set = 10000

outfile_name_MOUSE_train = basicdir + "/all_mfa_file_MOUSE_" + str(size_of_set) + "_train.fasta"
outfile_name_MOUSE_control = basicdir + "/all_mfa_file_MOUSE_" + str(size_of_set) + "_control.fasta"

outfile_MOUSE_train = open(outfile_name_MOUSE_train, "w") 
outfile_MOUSE_control = open(outfile_name_MOUSE_control, "w") 
log_files_to_mfa = open("log_files_to_mfa.txt", "w") 

headerStr_names_outfile_MOUSE_train = basicdir + "/headerStr_names_MOUSE_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_MOUSE_control = basicdir + "/headerStr_names_MOUSE_" + str(size_of_set) + "_control.csv"

def my_function(x):
    return list(dict.fromkeys(x))

headerStr_names_MOUSE_train = []
index_global = 0

for i in sorted(selected_MOUSE):
    print(i)
    tmp_file_list = []
    for j in files_mfa_MOUSE_train:
        if i in j:
            headerStr_base = "_".join(j.split(".")[:-1]).split("/")[-1]
            print(headerStr_base)
            with open(j, "r") as f:
                fiter = fasta_iter(f)
                index_local = 0
                for seq_rec in fiter:
                    headerStr_new = headerStr_base + "_globalInd_" + str(index_global) + "_localInd_" + str(index_local) + "_train"
                    headerStr, seq = seq_rec
                    seq = seq[350:650]
                    
                    if len(seq) < 320 and len(seq) > 280:
                        seq = seq[(len(seq)//2)-150:(len(seq)//2)+150]
                        if headerStr_base.split("_")[1] == organism:
                            tmp_file_list.append(">" + headerStr_new + "\n" + seq + "\n")
                        index_local += 1
            index_global += 1
    print("tmp_file_list", len(tmp_file_list))

    
    if len(tmp_file_list) == 0:
        continue

    print("tmp_file_list bewfore ", len(tmp_file_list))
    if len(tmp_file_list) > size_of_set:
        tmp_file_list = np.random.choice(tmp_file_list, size_of_set, replace=False, random_state=1)

    print("tmp_file_list after", len(tmp_file_list))
    log_files_to_mfa.write(i + " " + str(len(tmp_file_list)) + " " + organism +"_train")
    
    print(len(tmp_file_list))
    tmp_file_list = my_function(tmp_file_list)
    print(len(tmp_file_list))
    
    
    
    for k in tmp_file_list:
        if len(k) != 0:
            headerStr_names_MOUSE_train.append(k[1:].split("\n")[0])
            outfile_MOUSE_train.write(k)

    with open(headerStr_names_outfile_MOUSE_train, "w") as output:
        writer = csv.writer(output, lineterminator='\n')
        for val in headerStr_names_MOUSE_train:
            writer.writerow([val])  


###
headerStr_names_MOUSE_control = []
index_global = 0

for i in sorted(selected_MOUSE):
    print(i)
    tmp_file_list = []
    for j in files_mfa_MOUSE_test:
        if i in j:
            headerStr_base = "_".join(j.split(".")[:-1]).split("/")[-1]
            print(headerStr_base)
            with open(j, "r") as f:
                fiter = fasta_iter(f)
                index_local = 0
                for seq_rec in fiter:
                    headerStr_new = headerStr_base + "_globalInd_" + str(index_global) + "_localInd_" + str(index_local) + "_train"
                    headerStr, seq = seq_rec
                    seq = seq[350:650]
                    
                    if len(seq) < 320 and len(seq) > 280:
                        seq = seq[(len(seq)//2)-150:(len(seq)//2)+150]
                        if headerStr_base.split("_")[1] == organism:
                            tmp_file_list.append(">" + headerStr_new + "\n" + seq + "\n")
                        index_local += 1
            index_global += 1
    print("tmp_file_list", len(tmp_file_list))


    if len(tmp_file_list) == 0:
        continue

    print("tmp_file_list bewfore ", len(tmp_file_list))
    if len(tmp_file_list) > size_of_set:
        tmp_file_list = np.random.choice(tmp_file_list, size_of_set, replace=False, random_state=1)

        
    print("tmp_file_list after", len(tmp_file_list))
    log_files_to_mfa.write(i + " " + str(len(tmp_file_list)) + " " + organism +"_control")
    
    print(len(tmp_file_list))
    tmp_file_list = my_function(tmp_file_list)
    print(len(tmp_file_list))
    
    
    for k in tmp_file_list:
        #print(k)
        if len(k) != 0:
            headerStr_names_MOUSE_control.append(k[1:].split("\n")[0])
            outfile_MOUSE_control.write(k)

    with open(headerStr_names_outfile_MOUSE_control, "w") as output:
        writer = csv.writer(output, lineterminator='\n')
        for val in headerStr_names_MOUSE_control:
            writer.writerow([val])  

outfile_MOUSE_control.close()
outfile_MOUSE_train.close()

print("done!")

log_files_to_mfa.close()

In [ ]:
size_of_set = 10000

outfile_name_MOUSE_train = basicdir + "/all_mfa_file_MOUSE_SLIM_" + str(size_of_set) + "_train.fasta"
outfile_name_MOUSE_control = basicdir + "/all_mfa_file_MOUSE_SLIM_" + str(size_of_set) + "_control.fasta"

outfile_MOUSE_train = open(outfile_name_MOUSE_train, "w") 
outfile_MOUSE_control = open(outfile_name_MOUSE_control, "w") 
log_files_to_mfa = open("log_files_to_mfa.txt", "w") 

headerStr_names_outfile_MOUSE_train = basicdir + "/headerStr_names_MOUSE_SLIM_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_MOUSE_control = basicdir + "/headerStr_names_MOUSE_SLIM_" + str(size_of_set) + "_control.csv"

def my_function(x):
    return list(dict.fromkeys(x))

headerStr_names_MOUSE_train = []
index_global = 0

for i in sorted(selected_MOUSE):
    print(i)
    tmp_file_list = []
    for j in files_mfa_MOUSE_train:
        if i in j:
            headerStr_base = "_".join(j.split(".")[:-1]).split("/")[-1]
            print(headerStr_base)
            with open(j, "r") as f:
                fiter = fasta_iter(f)
                index_local = 0
                for seq_rec in fiter:
                    headerStr, seq = seq_rec
                    headerStr_new = headerStr_base + "_globalInd_" + str(index_global) + "_localInd_" + str(index_local) + "_train"
                    headerStr_new = headerStr_new + "$" + f'>chrom: {headerStr.split(":")[3]}; center: {headerStr.split(":")[1]}; peak: 501; signal: {headerStr.split(":")[0]}'
                    if True:
                        if headerStr_base.split("_")[1] == organism:
                            tmp_file_list.append(">" + headerStr_new + "\n" + seq + "\n")
                        index_local += 1
            index_global += 1
    print("tmp_file_list", len(tmp_file_list))

    
    if len(tmp_file_list) == 0:
        continue
        
    print("tmp_file_list bewfore ", len(tmp_file_list))
    if len(tmp_file_list) > size_of_set:
        tmp_file_list = np.random.choice(tmp_file_list, size_of_set, replace=False, random_state=1)

    print("tmp_file_list after", len(tmp_file_list))
    log_files_to_mfa.write(i + " " + str(len(tmp_file_list)) + " " + organism +"_train")
    
    print(len(tmp_file_list))
    tmp_file_list = my_function(tmp_file_list)
    print(len(tmp_file_list))
    
    for k in tmp_file_list:
        #print(k)
        if len(k) != 0:
            headerStr_names_MOUSE_train.append(k[1:].split("\n")[0])
            outfile_MOUSE_train.write(k)

    with open(headerStr_names_outfile_MOUSE_train, "w") as output:
        writer = csv.writer(output, lineterminator='\n')
        for val in headerStr_names_MOUSE_train:
            writer.writerow([val])  

###
headerStr_names_MOUSE_control = []
index_global = 0

for i in sorted(selected_MOUSE):
    print(i)
    tmp_file_list = []
    for j in files_mfa_MOUSE_test:
        if i in j:
            headerStr_base = "_".join(j.split(".")[:-1]).split("/")[-1]
            print(headerStr_base)
            with open(j, "r") as f:
                fiter = fasta_iter(f)
                index_local = 0
                for seq_rec in fiter:
                    headerStr, seq = seq_rec
                    headerStr_new = headerStr_base + "_globalInd_" + str(index_global) + "_localInd_" + str(index_local) + "_train"
                    headerStr_new = headerStr_new + "$" + f'>chrom: {headerStr.split(":")[3]}; center: {headerStr.split(":")[1]}; peak: 501; signal: {headerStr.split(":")[0]}'
                    if True:
                        if headerStr_base.split("_")[1] == organism:
                            tmp_file_list.append(">" + headerStr_new + "\n" + seq + "\n")
                        index_local += 1
            index_global += 1
    print("tmp_file_list", len(tmp_file_list))

    if len(tmp_file_list) == 0:
        continue

    print("tmp_file_list bewfore ", len(tmp_file_list))
    if len(tmp_file_list) > size_of_set:
        tmp_file_list = np.random.choice(tmp_file_list, size_of_set, replace=False, random_state=1)
        
    print("tmp_file_list after", len(tmp_file_list))
    log_files_to_mfa.write(i + " " + str(len(tmp_file_list)) + " " + organism +"_control")
    
    print(len(tmp_file_list))
    tmp_file_list = my_function(tmp_file_list)
    print(len(tmp_file_list))
    
    for k in tmp_file_list:
        if len(k) != 0:
            headerStr_names_MOUSE_control.append(k[1:].split("\n")[0])
            outfile_MOUSE_control.write(k)

    with open(headerStr_names_outfile_MOUSE_control, "w") as output:
        writer = csv.writer(output, lineterminator='\n')
        for val in headerStr_names_MOUSE_control:
            writer.writerow([val])  

outfile_MOUSE_control.close()
outfile_MOUSE_train.close()

print("done!")

log_files_to_mfa.close()

In [ ]:
outfile_name_MOUSE_train = basicdir + "/all_mfa_file_MOUSE_SLIM_" + str(size_of_set) + "_train.fasta"
outfile_name_MOUSE_control = basicdir + "/all_mfa_file_MOUSE_SLIM_" + str(size_of_set) + "_control.fasta"
outfile_name_HUMAN_train = basicdir + "/all_mfa_file_HUMAN_SLIM_" + str(size_of_set) + "_train.fasta"
outfile_name_HUMAN_control = basicdir + "/all_mfa_file_HUMAN_SLIM_" + str(size_of_set) + "_control.fasta"

In [ ]:
os.chdir(basicdir)

outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"

line = "awk 'NF' {outfile_name} > {outfile_name_no_NF}".format(outfile_name=outfile_name_HUMAN_train, outfile_name_no_NF=outfile_name_HUMAN_train_no_NF)
print(line)
p = subprocess.Popen(line, shell=True)
p.wait()

outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"

line = "awk 'NF' {outfile_name} > {outfile_name_no_NF}".format(outfile_name=outfile_name_MOUSE_train, outfile_name_no_NF=outfile_name_MOUSE_train_no_NF)
print(line)
p = subprocess.Popen(line, shell=True)
p.wait()

outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"

line = "awk 'NF' {outfile_name} > {outfile_name_no_NF}".format(outfile_name=outfile_name_HUMAN_control, outfile_name_no_NF=outfile_name_HUMAN_control_no_NF)
print(line)
p = subprocess.Popen(line, shell=True)
p.wait()

outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

line = "awk 'NF' {outfile_name} > {outfile_name_no_NF}".format(outfile_name=outfile_name_MOUSE_control, outfile_name_no_NF=outfile_name_MOUSE_control_no_NF)
print(line)
p = subprocess.Popen(line, shell=True)
p.wait()

os.chdir(outputdir)

In [ ]:
size_of_set = 10000
outfile_name_HUMAN_train = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_train.fasta"
outfile_name_MOUSE_train = basicdir + "/all_mfa_file_MOUSE_" + str(size_of_set) + "_train.fasta"
outfile_name_HUMAN_control = basicdir + "/all_mfa_file_HUMAN_" + str(size_of_set) + "_control.fasta"
outfile_name_MOUSE_control = basicdir + "/all_mfa_file_MOUSE_" + str(size_of_set) + "_control.fasta"

outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

In [ ]:
size_of_set = 10000
headerStr_names_outfile_HUMAN_train = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_MOUSE_train = basicdir + "/headerStr_names_MOUSE_" + str(size_of_set) + "_train.csv"
headerStr_names_outfile_HUMAN_control = basicdir + "/headerStr_names_HUMAN_" + str(size_of_set) + "_control.csv"
headerStr_names_outfile_MOUSE_control = basicdir + "/headerStr_names_MOUSE_" + str(size_of_set) + "_control.csv"

In [ ]:
out_tab_file_HUMAN_train = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_train.tab"
out_tab_file_MOUSE_train = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_train.tab"
out_tab_file_HUMAN_control = basicdir + "/" + "out_tab_HUMAN_" + str(size_of_set) + "_control.tab"
out_tab_file_MOUSE_control = basicdir + "/" + "out_tab_MOUSE_" + str(size_of_set) + "_control.tab"

In [ ]:
os.chdir(basicdir)

outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"

line = "awk 'NF' {outfile_name} > {outfile_name_no_NF}".format(outfile_name=outfile_name_HUMAN_train, outfile_name_no_NF=outfile_name_HUMAN_train_no_NF)
print(line)
p = subprocess.Popen(line, shell=True)
p.wait()

outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"

line = "awk 'NF' {outfile_name} > {outfile_name_no_NF}".format(outfile_name=outfile_name_MOUSE_train, outfile_name_no_NF=outfile_name_MOUSE_train_no_NF)
print(line)
p = subprocess.Popen(line, shell=True)
p.wait()

outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"

line = "awk 'NF' {outfile_name} > {outfile_name_no_NF}".format(outfile_name=outfile_name_HUMAN_control, outfile_name_no_NF=outfile_name_HUMAN_control_no_NF)
print(line)
p = subprocess.Popen(line, shell=True)
p.wait()

outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

line = "awk 'NF' {outfile_name} > {outfile_name_no_NF}".format(outfile_name=outfile_name_MOUSE_control, outfile_name_no_NF=outfile_name_MOUSE_control_no_NF)
print(line)
p = subprocess.Popen(line, shell=True)
p.wait()

os.chdir(outputdir)

In [ ]:
family_annotation = pd.read_csv("/home/pavelkrav/core_annotation_HUMAN_mono.csv", sep='\t')

def find_TF_family_dict(family_annotation):
    TF_family_dict = defaultdict(None)
    for i, j in zip(family_annotation["Model"], family_annotation["TF family"]):
        if type(j) == str:
            TF_family_dict[i.split('_')[0]] = j.split('{')[1][:3]
    return TF_family_dict

TF_family_dict = find_TF_family_dict(family_annotation)


In [ ]:
df_HUMAN = pd.DataFrame({'Names': headerStr_names_HUMAN_train})
a = []
for x in headerStr_names_HUMAN_train:
    try:
        a.append(TF_family_dict[x.split('_')[0]])
    except KeyError:
        a.append(None)
df_HUMAN["TF_family"] = np.array(a)  

names = ["_".join(x.split("_")[:2]) for x in headerStr_names_HUMAN_train]
df_HUMAN["TF_name"] = np.array(names)  
 
pd.DataFrame.to_csv(df_HUMAN, out_tab_file_HUMAN_train,  sep='\t', header=False)

###############
df_MOUSE = pd.DataFrame({'Names': headerStr_names_MOUSE_train})
a = []
for x in headerStr_names_MOUSE_train:
    try:
        a.append(TF_family_dict[x.split('_')[0]])
    except KeyError:
        a.append(None)
df_MOUSE["TF_family"] = np.array(a)  

names = ["_".join(x.split("_")[:2]) for x in headerStr_names_MOUSE_train]
df_MOUSE["TF_name"] = np.array(names)  
 
pd.DataFrame.to_csv(df_MOUSE, out_tab_file_MOUSE_train,  sep='\t', header=False)
print("ok")

###############
df_HUMAN = pd.DataFrame({'Names': headerStr_names_HUMAN_control})
a = []
for x in headerStr_names_HUMAN_control:
    try:
        a.append(TF_family_dict[x.split('_')[0]])
    except KeyError:
        a.append(None)
df_HUMAN["TF_family"] = np.array(a)  

names = ["_".join(x.split("_")[:2]) for x in headerStr_names_HUMAN_control]
df_HUMAN["TF_name"] = np.array(names)  
 
pd.DataFrame.to_csv(df_HUMAN, out_tab_file_HUMAN_control,  sep='\t', header=False)

###############
df_MOUSE = pd.DataFrame({'Names': headerStr_names_MOUSE_control})
a = []
for x in headerStr_names_MOUSE_control:
    try:
        a.append(TF_family_dict[x.split('_')[0]])
    except KeyError:
        a.append(None)
df_MOUSE["TF_family"] = np.array(a)  

names = ["_".join(x.split("_")[:2]) for x in headerStr_names_MOUSE_control]
df_MOUSE["TF_name"] = np.array(names)  
 
pd.DataFrame.to_csv(df_MOUSE, out_tab_file_MOUSE_control,  sep='\t', header=False)
print("ok")

In [ ]:
trimmed = False
size = 20
begin_trim = size
end_trim = size
if trimmed == True:
    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"

    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control_no_NF.split(".fasta")[0] + "_trimmed_{begin_trim}_{end_trim}.fasta".format(begin_trim=begin_trim, end_trim=end_trim)
else:
    outfile_name_HUMAN_train_no_NF = outfile_name_HUMAN_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_train_no_NF = outfile_name_MOUSE_train.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_HUMAN_control_no_NF = outfile_name_HUMAN_control.split(".fasta")[0]+"_no_NF.fasta"
    outfile_name_MOUSE_control_no_NF = outfile_name_MOUSE_control.split(".fasta")[0]+"_no_NF.fasta"


In [ ]:
with open(headerStr_names_outfile_HUMAN_train, "r") as output:
    headerStr_names_HUMAN_train = output.read().split('\n')
    print(headerStr_names_HUMAN_train[:10])

with open(headerStr_names_outfile_MOUSE_control, "r") as output:
    headerStr_names_MOUSE_control = output.read().split('\n')
    print(headerStr_names_MOUSE_control[:10])  
    
with open(headerStr_names_outfile_HUMAN_control, "r") as output:
    headerStr_names_HUMAN_train = output.read().split('\n')
    print(headerStr_names_HUMAN_train[:10])
    
with open(headerStr_names_outfile_MOUSE_train, "r") as output:
    headerStr_names_MOUSE_control = output.read().split('\n')
    print(headerStr_names_MOUSE_control[:10])  

In [ ]:
family_annotation = pd.read_csv("/home/pavelkrav/core_annotation_HUMAN_mono.csv", sep='\t')

def find_TF_family_dict(family_annotation):
    TF_family_dict = defaultdict(None)
    for i, j in zip(family_annotation["Model"], family_annotation["TF family"]):
        if type(j) == str:
            TF_family_dict[i.split('_')[0]] = j.split('{')[1][:3]
    return TF_family_dict

TF_family_dict = find_TF_family_dict(family_annotation)